# Experiment 2: Compare Loss Functions, Optimizers, and Regularization

This practical uses a feed-forward network on the breast cancer dataset and compares:
- loss: `binary_crossentropy`, `mse`
- optimizer: `adam`, `sgd`
- regularization: none, dropout, L2
        

In [1]:
!pip install numpy pandas scikit-learn "tensorflow-cpu>=2.15,<2.18"

  Using cached tensorflow_cpu-2.17.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached ml_dtypes-0.4.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached tensorboard-2.17.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
Using cached tensorflow_cpu-2.17.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (221.2 MB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached ml_dtypes-0.4.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_6

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
        

In [ ]:
def build_model(input_dim, reg=None):
    model = Sequential([Input(shape=(input_dim,))])

    if reg == "l2":
        model.add(Dense(16, activation="relu", kernel_regularizer=l2(0.01)))
    else:
        model.add(Dense(16, activation="relu"))

    if reg == "dropout":
        model.add(Dropout(0.3))

    model.add(Dense(1, activation="sigmoid"))
    return model

configs = [
    ("binary_crossentropy", "adam", None),
    ("binary_crossentropy", "sgd", None),
    ("mse", "adam", None),
    ("binary_crossentropy", "adam", "dropout"),
    ("binary_crossentropy", "adam", "l2"),
]

rows = []
for loss_name, optimizer_name, reg in configs:
    model = build_model(X_train.shape[1], reg=reg)
    model.compile(optimizer=optimizer_name, loss=loss_name, metrics=["accuracy"])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    _, acc = model.evaluate(X_test, y_test, verbose=0)

    rows.append(
        {
            "loss": loss_name,
            "optimizer": optimizer_name,
            "regularization": reg if reg else "none",
            "accuracy": float(acc),
        }
    )

result_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
result_df